# KDD vs CIC-IDS-2017 入侵检测对比实验

**Colab 推荐**：`CIC_DOWNLOAD_MODE=hf_minimal`（Hugging Face 按日 parquet，约 55MB）

按顺序执行。KDD 由 sklearn 自动下载；CIC 无需本机存全量数据。

In [ ]:
# 1. 环境与依赖
import os, sys, platform
from pathlib import Path


# CIC 下载：hf_minimal(推荐,~55MB) | hf_one | drive | zip | bundled
os.environ.setdefault("CIC_DOWNLOAD_MODE", "hf_minimal")
# QUICK_RUN: 调试时取消下一行注释
# os.environ["QUICK_RUN"] = "true"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/intrusion-detection-kdd-cic")
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT = Path("/content/intrusion-detection-kdd-cic")
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

if IN_COLAB:
    get_ipython().system(f'pip install -q -r "{PROJECT_ROOT}/requirements-colab.txt"')

import psutil
print("RAM (GB):", round(psutil.virtual_memory().total / 1e9, 1))
print("Platform:", platform.platform())
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
# 2. 克隆/同步项目（若 Drive 中无代码）
if IN_COLAB and not (PROJECT_ROOT / "src").exists():
    get_ipython().system('git clone https://github.com/Wentao900/intrusion-detection-kdd-cic.git /content/intrusion-detection-kdd-cic')
    PROJECT_ROOT = Path("/content/intrusion-detection-kdd-cic")
    os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
    sys.path.insert(0, str(PROJECT_ROOT))


## （一）数据集深度分析

In [ ]:
# 3. 加载 KDD
from src.data_kdd import load_kdd_pipeline
from src.eda import run_eda_kdd, plot_mutual_information, plot_correlation_heatmap
from src.preprocess import preprocess_dataset, imbalance_ratio
from src.config import ARTIFACTS_DIR
import pandas as pd
import numpy as np

kdd_df, kdd_meta = load_kdd_pipeline()
print("KDD meta:", kdd_meta)
eda_kdd = run_eda_kdd(kdd_df)
print("EDA KDD:", eda_kdd["stats"])

kdd_pp = preprocess_dataset(kdd_df, "label_category", "kdd")
print("KDD features:", kdd_pp.X_train.shape[1], "imbalance:", imbalance_ratio(kdd_pp.y_train))
plot_mutual_information(kdd_pp.X_train, kdd_pp.y_train, kdd_pp.feature_names,
    "KDD MI", ARTIFACTS_DIR / "eda_kdd" / "mutual_information.png")
Xdf = pd.DataFrame(kdd_pp.X_train, columns=kdd_pp.feature_names)
plot_correlation_heatmap(Xdf, "KDD Corr", ARTIFACTS_DIR / "eda_kdd" / "correlation.png")


In [ ]:
# 4. 加载 CIC（HF 按日分片 ~55MB，无需本机 224MB zip）
# 若失败: os.environ['CIC_DOWNLOAD_MODE']='bundled'
from src.data_cic import load_cic_chunked
from src.eda import run_eda_cic

cic_df, cic_meta = load_cic_chunked()
print("CIC meta:", cic_meta)
eda_cic = run_eda_cic(cic_df)
print("EDA CIC:", eda_cic["stats"])

cic_pp = preprocess_dataset(cic_df, "Label", "cic")
print("CIC features:", cic_pp.X_train.shape[1])


## （二）机器学习模型构建与对比

In [ ]:
# 5. 经典 ML + CNN 训练
from src.models_ml import ML_MODEL_NAMES, train_and_evaluate_ml
from src.models_dl import train_and_evaluate_cnn
from src.eda import plot_confusion_matrix
import json

all_metrics = []
for name in ML_MODEL_NAMES:
    m = train_and_evaluate_ml(name, kdd_pp, "kdd")
    all_metrics.append(m)
    print(f"KDD {name}: F1={m['f1_macro']:.4f}")
for name in ML_MODEL_NAMES:
    m = train_and_evaluate_ml(name, cic_pp, "cic")
    all_metrics.append(m)
    print(f"CIC {name}: F1={m['f1_macro']:.4f}")

cnn_k = train_and_evaluate_cnn(kdd_pp, "kdd")
cnn_c = train_and_evaluate_cnn(cic_pp, "cic")
all_metrics.extend([cnn_k, cnn_c])
print("Done ML+CNN")


## （三）模型优化与创新

In [ ]:
# 6. 集成 / SMOTE / 分层创新 / 跨数据集
from src.optimize import (
    train_ensemble, train_with_smote,
    train_hierarchical_stack, cross_dataset_transfer_experiment,
)

opt_results = []
opt_results.append(train_ensemble(kdd_pp, "kdd"))
opt_results.append(train_ensemble(cic_pp, "cic"))
opt_results.append(train_with_smote(cic_pp, "cic"))

innov_results = []
innov_results.append(train_hierarchical_stack(kdd_pp, "kdd"))
innov_results.append(train_hierarchical_stack(cic_pp, "cic"))

transfer = cross_dataset_transfer_experiment(kdd_pp, cic_pp)
print("Transfer:", transfer)


In [ ]:
# 7. 保存 metrics 并生成报告
from src.run_experiment import _env_info, _serialize_results
from src.config import ARTIFACTS_DIR
from pathlib import Path
import json

results = {
    "env": _env_info(),
    "eda": {"kdd": eda_kdd, "cic": eda_cic},
    "kdd_meta": kdd_meta,
    "cic_meta": cic_meta,
    "preprocess": {"kdd": kdd_pp.meta, "cic": cic_pp.meta},
    "models": all_metrics,
    "optimization": opt_results,
    "innovation": innov_results,
    "transfer": transfer,
}
metrics_path = ARTIFACTS_DIR / "metrics.json"
_serialize_results(results, metrics_path)
print("Saved:", metrics_path)


In [ ]:
# 8. 生成中文实验报告
from src.report_builder import build_report
report_path = build_report()
print("Report:", report_path)
# 在 Colab 中下载报告
if IN_COLAB:
    from google.colab import files
    files.download(str(report_path))
